# StairKid RL — self-contained V3 / R4 training

This is the primary human-facing training workflow. Simulator, observation,
reward, and curriculum environment implementations stay in the installed
`stair_agent` package; the complete PPO orchestration is visible below:
configuration, assets, resume validation, environment/PPO construction,
curriculum transitions, evaluation, checkpointing, manifests, precheck,
smoke, and full learning.

**Safety:** the default is `precheck`. Full training requires the exact
authorization phrase. This notebook never connects to the Real game and
never promotes or overwrites a canonical model.

## 1. User configuration

Normally change only this cell. Use `stairkid-final` or an exact commit SHA
instead of `main` when reproducing a frozen run. Smoke is always eight
simulator steps; full mode uses the canonical YAML target totals.

In [ ]:
TRAIN_TARGET = "v3"  # exactly: v3 or r4
TRAINING_MODE = "precheck"  # precheck, smoke, or full

REPO_URL = "https://github.com/GameToy21452244/stairkid-rl.git"
GIT_REF = "main"  # main, stairkid-final, another tag, or an exact commit SHA

SEED = None  # None selects the preset default (v3=17, r4=142)
DEVICE = None  # None selects the preset value; "auto", "cpu", or "cuda" are allowed

RESUME = False
RESUME_CHECKPOINT = None
RESUME_METADATA = None

OUTPUT_TO_DRIVE = True
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/StairKidRL/runs"
TRAINING_ASSET_SOURCE_DIR = None  # directory containing the pinned R4 bundle
MODEL_ASSET_SOURCE_DIR = None  # directory containing both canonical model archives

ALLOW_DIRTY = False
AUTHORIZATION = ""  # full requires AUTHORIZE_STAIRKID_FULL_TRAINING

assert TRAIN_TARGET in {"v3", "r4"}
assert TRAINING_MODE in {"precheck", "smoke", "full"}
assert not RESUME or RESUME_CHECKPOINT is not None

## 2. Colab / repository setup

Git is the source of truth. Colab clones and checks out the requested ref,
then installs the reusable environment package. Git checkout is the only
source workflow, and no `PYTHONPATH` workaround is used.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    WORKDIR = Path("/content/stairkid-rl")
    if WORKDIR.exists():
        shutil.rmtree(WORKDIR)
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
    subprocess.run(["git", "checkout", GIT_REF], cwd=WORKDIR, check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", ".[rl]"],
        cwd=WORKDIR,
        check=True,
    )
else:
    candidate = Path.cwd().resolve()
    WORKDIR = candidate if (candidate / "pyproject.toml").is_file() else candidate.parent
    if not (WORKDIR / "pyproject.toml").is_file():
        raise RuntimeError("LOCAL_NOTEBOOK_MUST_RUN_FROM_REPOSITORY")

os.chdir(WORKDIR)
PROJECT_ROOT = WORKDIR.resolve()
RESOLVED_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()

import stair_agent

print(f"REPOSITORY={REPO_URL}")
print(f"GIT_REF={GIT_REF}")
print(f"COMMIT={RESOLVED_COMMIT}")
print(f"TRAIN_TARGET={TRAIN_TARGET}")
print(f"PYTHON={sys.version.split()[0]}")
print("PACKAGE_IMPORT=PASS")

## 3. Imports, versions, and output location

Drive stores outputs only. Repository source remains the checked-out Git
tree, while models and training inputs remain separately verified assets.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timezone
import hashlib
import json
import platform
from pathlib import PurePosixPath
from typing import Any, Callable, Mapping
from urllib.request import urlopen
import zipfile

import numpy as np
import stable_baselines3
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
import torch
import yaml

from stair_agent.core.model_registry import load_model_registry
from stair_agent.envs.fidelity_v3_fresh import make_fidelity_v3_fresh_env
from stair_agent.envs.fidelity_v3_5 import make_fidelity_v3_5_env
from stair_agent.evaluation import floor_metrics
from stair_agent.fresh_v3_curriculum import (
    FreshV3SelfCurriculumEnv,
    collect_self_curriculum_bank,
)
from stair_agent.training.configs import TARGET_IDS, load_training_target
from stair_agent.v3_5_curriculum import (
    V35TargetedCurriculumEnv,
    validate_v35_targeted_bank,
)

if OUTPUT_TO_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("OUTPUT_TO_DRIVE_REQUIRES_COLAB")
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path(DRIVE_OUTPUT_ROOT).resolve()
else:
    OUTPUT_ROOT = (PROJECT_ROOT / "runs").resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"TORCH={torch.__version__}")
print(f"CUDA_AVAILABLE={torch.cuda.is_available()}")
print(f"CUDA_VERSION={torch.version.cuda}")
print(f"GPU_NAME={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"SB3_VERSION={stable_baselines3.__version__}")
print(f"OUTPUT_ROOT={OUTPUT_ROOT}")

## 4. Canonical target and artifact identities

YAML remains the canonical preset data, but every value consumed by PPO is
mapped explicitly below. The only active external SHA gates are the two
canonical model archives and the one R4 bundle SHA read from its manifest.

In [ ]:
FULL_TRAINING_AUTHORIZATION = "AUTHORIZE_STAIRKID_FULL_TRAINING"
SMOKE_TIMESTEPS = 8
EXPECTED_OBSERVATION_SHAPE = (268,)
EXPECTED_ACTION_COUNT = 3
EXPECTED_MODEL_SHA = {
    "v3": "e539ad8e9a39991d738ef9d4113968d933d4f2535e3b08fabe27f3b4ffd9f51e",
    "r4": "6a9e966ae69c1b3f5610bc5c8a009dcc5519e94fa20d754e54ef0ac445399e10",
}

if tuple(TARGET_IDS) != ("v3", "r4"):
    raise RuntimeError("TRAINING_TARGET_REGISTRY_CHANGED")
TARGET = load_training_target(PROJECT_ROOT, TRAIN_TARGET)
MODEL_REGISTRY = load_model_registry(PROJECT_ROOT)
if tuple(MODEL_REGISTRY) != ("v3", "r4"):
    raise RuntimeError("MODEL_REGISTRY_CHANGED")
for model_id, expected_sha in EXPECTED_MODEL_SHA.items():
    if MODEL_REGISTRY[model_id].sha256 != expected_sha:
        raise RuntimeError(f"CANONICAL_MODEL_PIN_CHANGED:{model_id}")

TRAINING_ASSET_MANIFEST_PATH = PROJECT_ROOT / "training_assets/manifest.json"
TRAINING_ASSET_MANIFEST = json.loads(
    TRAINING_ASSET_MANIFEST_PATH.read_text(encoding="utf-8")
)
if tuple(TRAINING_ASSET_MANIFEST["assets"]) != ("r4_frozen_r1_bundle",):
    raise RuntimeError("TRAINING_ASSET_REGISTRY_CHANGED")
R4_BUNDLE_SPEC = TRAINING_ASSET_MANIFEST["assets"]["r4_frozen_r1_bundle"]
R4_BUNDLE_SHA = str(R4_BUNDLE_SPEC["sha256"])

RUN_SEED = TARGET.default_seed if SEED is None else int(SEED)
RUN_DEVICE = str(TARGET.algorithm["device"]) if DEVICE is None else str(DEVICE)
PROFILE_PATH = (PROJECT_ROOT / str(TARGET.environment["profile"])).resolve()

print(f"TARGET={TARGET.id}")
print(f"SEED={RUN_SEED}")
print(f"DEVICE={RUN_DEVICE}")
print(f"CONFIG_SHA256={TARGET.config_sha256}")
print(f"OBSERVATION_SPACE={EXPECTED_OBSERVATION_SHAPE}")
print(f"ACTION_SPACE=Discrete({EXPECTED_ACTION_COUNT})")

## 5. Artifact discovery, copy/download, and SHA verification

Assets are copied through a `.partial` file and promoted only after SHA
verification. There is no similar-model fallback. Resume checkpoints do
**not** require a user-supplied SHA; their SHA is computed as provenance.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def copy_or_download_verified(
    *, filename: str, destination: Path, expected_sha: str,
    source_dir: str | None, remote_url: str | None, label: str,
) -> Path:
    destination = destination.resolve()
    if destination.is_file():
        actual = sha256_file(destination)
        if actual != expected_sha:
            raise RuntimeError(f"{label}_CACHE_SHA_MISMATCH:{actual}")
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix + ".partial")
    partial.unlink(missing_ok=True)
    if source_dir:
        root = Path(source_dir).expanduser().resolve()
        source = (root / filename).resolve()
        if not source.is_relative_to(root) or not source.is_file():
            raise RuntimeError(f"{label}_SOURCE_FILE_MISSING:{source}")
        shutil.copyfile(source, partial)
    elif remote_url:
        with urlopen(remote_url, timeout=60) as response, partial.open("wb") as output:
            shutil.copyfileobj(response, output)
    else:
        raise RuntimeError(f"{label}_REMOTE_UNPUBLISHED; provide a source directory")
    actual = sha256_file(partial)
    if actual != expected_sha:
        partial.unlink(missing_ok=True)
        raise RuntimeError(f"{label}_SHA_MISMATCH:{actual}!={expected_sha}")
    partial.replace(destination)
    return destination


def ensure_canonical_model(model_id: str) -> Path:
    spec = MODEL_REGISTRY[model_id]
    return copy_or_download_verified(
        filename=spec.asset_path.name,
        destination=spec.asset_path,
        expected_sha=spec.sha256,
        source_dir=MODEL_ASSET_SOURCE_DIR,
        remote_url=spec.metadata.get("release_url"),
        label=f"MODEL_{model_id.upper()}",
    )


def ensure_r4_bundle() -> Path:
    cache_path = (PROJECT_ROOT / str(R4_BUNDLE_SPEC["cache_path"])).resolve()
    return copy_or_download_verified(
        filename=str(R4_BUNDLE_SPEC["filename"]),
        destination=cache_path,
        expected_sha=R4_BUNDLE_SHA,
        source_dir=TRAINING_ASSET_SOURCE_DIR,
        remote_url=R4_BUNDLE_SPEC.get("source", {}).get("url"),
        label="R4_TRAINING_BUNDLE",
    )

## 6. Preset validation and environment factories

These checks make the YAML-to-PPO mapping explicit. Both human-readable
targets use the same corrected simulator package, `(268,)` observations,
`Discrete(3)` actions, and 60 Hz physics with 8/10/12 Hz policy cadence.

In [ ]:
EXPECTED_PPO = {
    "name": "PPO", "policy": "MlpPolicy", "n_envs": 4,
    "n_steps": 1024, "batch_size": 256, "n_epochs": 10,
    "learning_rate": 0.0003, "gamma": 0.99, "gae_lambda": 0.95,
    "clip_range": 0.2, "ent_coef": 0.01, "vf_coef": 0.5,
    "max_grad_norm": 0.5,
}
for key, expected in EXPECTED_PPO.items():
    if TARGET.algorithm[key] != expected:
        raise RuntimeError(f"PPO_PRESET_MISMATCH:{key}")
if TARGET.observation_shape != EXPECTED_OBSERVATION_SHAPE:
    raise RuntimeError("OBSERVATION_CONTRACT_MISMATCH")
if TARGET.action_count != EXPECTED_ACTION_COUNT:
    raise RuntimeError("ACTION_CONTRACT_MISMATCH")
if int(TARGET.environment["physics_hz"]) != 60:
    raise RuntimeError("PHYSICS_HZ_CONTRACT_MISMATCH")
if TARGET.id == "r4" and float(TARGET.raw["reward"]["edge_landing_penalty"]) != 1.10:
    raise RuntimeError("R4_EDGE_LANDING_PENALTY_MISMATCH")


def ordinary_factory(target_id: str, profile: Path, base_seed: int) -> Callable[[], Any]:
    if target_id == "v3":
        return lambda: make_fidelity_v3_fresh_env(profile, base_seed=base_seed)
    return lambda: make_fidelity_v3_5_env(profile, base_seed=base_seed)


def ordinary_vec(target: Any, profile: Path, seed: int, *, smoke: bool):
    count = 1 if smoke else int(target.algorithm["n_envs"])
    return DummyVecEnv([
        ordinary_factory(
            target.id, profile,
            1_000_000 + seed * 100_000 + index * 10_000_000,
        )
        for index in range(count)
    ])


def v3_curriculum_vec(
    profile: Path, seed: int, bank_dir: Path, *, n_envs: int, stage_offset: int,
):
    manifest = bank_dir / "manifest.json"
    return DummyVecEnv([
        lambda index=index: FreshV3SelfCurriculumEnv(
            profile_path=profile,
            bank_manifest_path=manifest,
            bank_root=bank_dir,
            base_seed=50_000_000 + seed * 100_000 + stage_offset + index * 10_000_000,
        )
        for index in range(n_envs)
    ])


def r4_curriculum_vec(
    profile: Path, seed: int, bank_dir: Path, source_sha: str, *, n_envs: int,
):
    if n_envs != 4:
        raise RuntimeError("R4_FIXED_VECTOR_LANES_REQUIRE_FOUR_ENVS")
    manifest = bank_dir / "manifest.json"
    modes = ("ordinary", "ordinary", "failure", "success")
    factories = []
    for index, mode in enumerate(modes):
        base_seed = 70_000_000 + seed * 100_000 + index * 10_000_000
        if mode == "ordinary":
            factories.append(
                lambda base_seed=base_seed: make_fidelity_v3_5_env(
                    profile, base_seed=base_seed
                )
            )
        else:
            factories.append(
                lambda base_seed=base_seed, mode=mode: V35TargetedCurriculumEnv(
                    profile_path=profile,
                    bank_root=bank_dir,
                    bank_manifest_path=manifest,
                    base_seed=base_seed,
                    expected_policy_seed=seed,
                    expected_source_sha256=source_sha,
                    fixed_mode=mode,
                    expected_source_timesteps=589_824,
                )
            )
    return DummyVecEnv(factories)

## 7. PPO factory

The actual SB3 construction is intentionally visible. Smoke changes only
rollout/batch/epoch sizes so the path completes in eight simulator steps;
full mode consumes the canonical preset values directly.

In [ ]:
def create_ppo_model(target: Any, env: Any, seed: int, device: str, *, smoke: bool):
    algorithm = target.algorithm
    model = PPO(
        str(algorithm["policy"]),
        env,
        learning_rate=float(algorithm["learning_rate"]),
        n_steps=8 if smoke else int(algorithm["n_steps"]),
        batch_size=8 if smoke else int(algorithm["batch_size"]),
        n_epochs=1 if smoke else int(algorithm["n_epochs"]),
        gamma=float(algorithm["gamma"]),
        gae_lambda=float(algorithm["gae_lambda"]),
        clip_range=float(algorithm["clip_range"]),
        ent_coef=float(algorithm["ent_coef"]),
        vf_coef=float(algorithm["vf_coef"]),
        max_grad_norm=float(algorithm["max_grad_norm"]),
        seed=seed,
        device=device,
        verbose=1,
    )
    return model

## 8. Resume checkpoint validation

A local checkpoint SHA is computed for the output manifest, not requested
from the user. The checkpoint must load into the target environment, match
target/config metadata, use `(268,)` / `Discrete(3)`, start at an allowed
timestep, and leave a valid amount of work.

In [ ]:
@dataclass(frozen=True)
class ResumeState:
    path: Path
    sha256: str
    model: Any
    current_timesteps: int
    target_timesteps: int
    remaining_timesteps: int


def validate_resume_checkpoint(
    checkpoint: Path,
    target: Any,
    *,
    env: Any,
    metadata_path: Path | None,
    device: str,
    allow_pinned_external: bool = False,
) -> ResumeState:
    path = checkpoint.expanduser().resolve()
    if not path.is_file():
        raise RuntimeError(f"RESUME_CHECKPOINT_REQUIRED:{path}")
    checkpoint_sha = sha256_file(path)  # provenance only; no RESUME_SHA input

    if metadata_path is not None:
        metadata = json.loads(metadata_path.expanduser().read_text(encoding="utf-8"))
        if metadata.get("training_target") != target.id:
            raise RuntimeError("RESUME_TARGET_MISMATCH")
        if metadata.get("config_sha256") != target.config_sha256:
            raise RuntimeError("RESUME_CONFIG_MISMATCH")
    elif target.raw["resume"]["require_matching_target_and_config_metadata"] and not allow_pinned_external:
        raise RuntimeError("RESUME_METADATA_REQUIRED")

    try:
        model = PPO.load(path, env=env, device=device)
    except (ValueError, TypeError) as exc:
        raise RuntimeError("RESUME_MODEL_ENVIRONMENT_INCOMPATIBLE") from exc
    if tuple(model.observation_space.shape) != EXPECTED_OBSERVATION_SHAPE:
        raise RuntimeError("RESUME_OBSERVATION_SPACE_MISMATCH")
    if int(model.action_space.n) != EXPECTED_ACTION_COUNT:
        raise RuntimeError("RESUME_ACTION_SPACE_MISMATCH")

    current = int(model.num_timesteps)
    allowed = tuple(int(value) for value in target.raw["resume"]["allowed_initial_timesteps"])
    if current not in allowed and not allow_pinned_external:
        raise RuntimeError(f"RESUME_TIMESTEPS_NOT_ALLOWED:{current}")
    if current >= target.total_timesteps:
        if current > target.total_timesteps or not allow_pinned_external:
            raise RuntimeError(f"RESUME_TARGET_NOT_AHEAD:{current}>={target.total_timesteps}")
    remaining = max(0, target.total_timesteps - current)
    return ResumeState(path, checkpoint_sha, model, current, target.total_timesteps, remaining)

## 9. R4 bundle validation and fixed curriculum lanes

The bundle archive SHA is the sole external R4 asset gate. After that gate,
this cell visibly checks ZIP integrity, safe paths, embedded contract,
seed/checkpoint pairing, checkpoint ZIPs, bank schema/snapshots, and the
checkpoint-to-bank identity. Child hashes are structural metadata—not
separate user-managed pins. R1 retraining and bank recollection remain
forbidden.

In [ ]:
def safe_zip_member(name: str) -> PurePosixPath:
    member = PurePosixPath(name)
    if member.is_absolute() or not member.parts or ".." in member.parts:
        raise RuntimeError(f"R4_BUNDLE_UNSAFE_ZIP_MEMBER:{name}")
    return member


def stage_and_validate_r4_bundle() -> Path:
    bundle = ensure_r4_bundle()
    if sha256_file(bundle) != R4_BUNDLE_SHA:
        raise RuntimeError("R4_BUNDLE_SHA_MISMATCH")
    stage_root = (PROJECT_ROOT / "training_assets/cache/r4").resolve()
    stage_root.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(bundle) as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"R4_BUNDLE_CRC_FAIL:{bad_member}")
        bundle_manifest = json.loads(archive.read("FROZEN_R1_BUNDLE_MANIFEST.json"))
        expected_contract = {
            "schema_version": "stairkid-v3-5-r4-frozen-r1-input-v1",
            "purpose": "R4_FROZEN_R1_AND_TARGETED_BANK_REUSE_ONLY",
            "policy_seeds": [117, 142],
            "r1_timesteps": 589824,
            "r1_retraining_forbidden": True,
            "bank_recollection_forbidden": True,
            "bank_schema_version": "v3-5-targeted-safety-bank-r3-corrected-flipping-v1",
            "bank_counts_per_seed": {"landing": 20, "spike": 20, "top": 8, "success": 48},
        }
        for key, expected in expected_contract.items():
            if bundle_manifest.get(key) != expected:
                raise RuntimeError(f"R4_BUNDLE_CONTRACT_MISMATCH:{key}")

        for name in archive.namelist():
            member = safe_zip_member(name)
            destination = (stage_root / Path(*member.parts)).resolve()
            if not destination.is_relative_to(stage_root):
                raise RuntimeError(f"R4_BUNDLE_MEMBER_ESCAPES_CACHE:{name}")
            if name.endswith("/"):
                destination.mkdir(parents=True, exist_ok=True)
                continue
            destination.parent.mkdir(parents=True, exist_ok=True)
            data = archive.read(name)
            if destination.is_file() and destination.read_bytes() == data:
                continue
            partial = destination.with_suffix(destination.suffix + ".partial")
            partial.write_bytes(data)
            partial.replace(destination)

    for seed in (117, 142):
        checkpoint = stage_root / f"seed_{seed}/checkpoints/v3_5_589824.zip"
        checkpoint_metadata = checkpoint.with_suffix(".json")
        bank_dir = stage_root / f"banks/seed_{seed}/r1_targeted"
        bank_manifest = bank_dir / "manifest.json"
        if not checkpoint.is_file() or not checkpoint_metadata.is_file():
            raise RuntimeError(f"R4_BUNDLE_CHECKPOINT_MISSING:{seed}")
        if not bank_manifest.is_file():
            raise RuntimeError(f"R4_BUNDLE_BANK_MISSING:{seed}")

        row = json.loads(checkpoint_metadata.read_text(encoding="utf-8"))
        checkpoint_sha = sha256_file(checkpoint)
        expected_path = f"seed_{seed}/checkpoints/v3_5_589824.zip"
        if (
            row.get("policy_seed") != seed
            or row.get("num_timesteps") != 589824
            or row.get("target_timesteps") != 589824
            or row.get("path") != expected_path
            or row.get("sha256") != checkpoint_sha
        ):
            raise RuntimeError(f"R4_CHECKPOINT_PAIRING_INVALID:{seed}")
        with zipfile.ZipFile(checkpoint) as checkpoint_archive:
            if checkpoint_archive.testzip() is not None:
                raise RuntimeError(f"R4_CHECKPOINT_CRC_FAIL:{seed}")

        # Keep the important bank integrity logic readable here instead
        # of making the notebook reader trust an opaque validator call.
        bank_row = json.loads(bank_manifest.read_text(encoding="utf-8"))
        if bank_row.get("schema_version") != expected_contract["bank_schema_version"]:
            raise RuntimeError(f"R4_BANK_SCHEMA_MISMATCH:{seed}")
        if int(bank_row.get("policy_seed", -1)) != seed:
            raise RuntimeError(f"R4_BANK_POLICY_SEED_MISMATCH:{seed}")
        if bank_row.get("source_model_sha256") != checkpoint_sha:
            raise RuntimeError(f"R4_BANK_SOURCE_SHA_MISMATCH:{seed}")
        if int(bank_row.get("source_model_timesteps", -1)) != 589824:
            raise RuntimeError(f"R4_BANK_SOURCE_TIMESTEPS_MISMATCH:{seed}")

        counts = {"landing": 0, "spike": 0, "top": 0, "success": 0}
        for entry in bank_row.get("entries", []):
            category = str(entry.get("category"))
            if category not in counts:
                raise RuntimeError(f"R4_BANK_CATEGORY_INVALID:{seed}:{category}")
            snapshot_path = (bank_dir / str(entry.get("snapshot_path"))).resolve()
            if not snapshot_path.is_relative_to(bank_dir.resolve()) or not snapshot_path.is_file():
                raise RuntimeError(f"R4_BANK_SNAPSHOT_MISSING:{seed}:{snapshot_path}")
            if sha256_file(snapshot_path) != entry.get("snapshot_sha256"):
                raise RuntimeError(f"R4_BANK_SNAPSHOT_SHA_MISMATCH:{seed}:{snapshot_path.name}")
            snapshot = json.loads(snapshot_path.read_text(encoding="utf-8"))
            if snapshot.get("schema_version") != "fidelity-v3-snapshot-v1":
                raise RuntimeError(f"R4_BANK_SNAPSHOT_SCHEMA_MISMATCH:{seed}")
            if entry.get("source_model_sha256") != checkpoint_sha:
                raise RuntimeError(f"R4_BANK_ENTRY_SOURCE_SHA_MISMATCH:{seed}")
            if int(entry.get("policy_seed", -1)) != seed:
                raise RuntimeError(f"R4_BANK_ENTRY_POLICY_SEED_MISMATCH:{seed}")
            if int(entry.get("source_model_timesteps", -1)) != 589824:
                raise RuntimeError(f"R4_BANK_ENTRY_TIMESTEPS_MISMATCH:{seed}")
            counts[category] += 1
        if counts != expected_contract["bank_counts_per_seed"]:
            raise RuntimeError(f"R4_BANK_COUNTS_INCOMPLETE:{seed}:{counts}")
        if bank_row.get("actual_counts") != counts or bank_row.get("status") != "PASS":
            raise RuntimeError(f"R4_BANK_MANIFEST_STATUS_INVALID:{seed}")

        # The package validator remains a second parity check for the
        # full frozen curriculum contract and episode-diversity rules.
        validate_v35_targeted_bank(
            bank_dir,
            bank_manifest,
            expected_policy_seed=seed,
            expected_source_sha256=checkpoint_sha,
            expected_source_timesteps=589824,
        )
    return stage_root

## 10. V3 self-curriculum transitions

Fresh V3 starts with ordinary episodes. At 196608 and 393216 steps the
current policy collects balanced failure/success snapshots, then switches
to the exact Stage B and Stage C schedules from the canonical preset.

In [ ]:
def activate_v3_curriculum_at_boundary(
    *, model: Any, current: int, run_dir: Path, seed: int,
    profile: Path, target: Any, resume_checkpoint: Path | None,
):
    boundaries = {
        196_608: {
            "bank_name": "stage_a_to_b", "stage_index": 1,
            "collection_base": 20_000_000, "stage_offset": 0,
        },
        393_216: {
            "bank_name": "stage_b_to_c", "stage_index": 2,
            "collection_base": 25_000_000, "stage_offset": 5_000_000,
        },
    }
    spec = boundaries.get(int(current))
    if spec is None:
        return None
    checkpoint_path = run_dir / "checkpoints" / f"fresh_v3_{current}.zip"
    if not checkpoint_path.is_file() and resume_checkpoint is not None:
        checkpoint_path = resume_checkpoint.resolve()
    if not checkpoint_path.is_file():
        raise RuntimeError(f"V3_CURRICULUM_SOURCE_CHECKPOINT_MISSING:{checkpoint_path}")

    bank_dir = run_dir / "banks" / f"seed_{seed}" / spec["bank_name"]
    stage = target.training["stages"][spec["stage_index"]]
    collect_self_curriculum_bank(
        model=model,
        profile_path=profile,
        output_dir=bank_dir,
        policy_seed=seed,
        source_model_sha256=sha256_file(checkpoint_path),
        source_model_timesteps=current,
        collection_seed_base=spec["collection_base"] + seed * 100_000,
        stage_label=f"seed_{seed}_{spec['bank_name']}",
        schedule_cycle=tuple(stage["schedule"]),
        target_per_class=int(target.raw["curriculum"]["target_per_class"]),
        max_episodes=int(target.raw["curriculum"]["collection_max_episodes"]),
        failure_lookback_steps=int(target.raw["curriculum"]["failure_snapshot_lookback_steps"]),
        success_min_floor=int(target.raw["curriculum"]["success_min_floor"]),
        success_snapshots_max_per_episode=int(target.raw["curriculum"]["success_snapshots_max_per_episode"]),
    )
    return v3_curriculum_vec(
        profile, seed, bank_dir, n_envs=4, stage_offset=spec["stage_offset"]
    )


def restore_v3_curriculum_for_resume(
    *, checkpoint: Path, start: int, profile: Path, seed: int,
):
    resume_run = checkpoint.resolve().parent.parent
    if 196_608 < start < 393_216:
        bank_dir = resume_run / "banks" / f"seed_{seed}" / "stage_a_to_b"
        return v3_curriculum_vec(profile, seed, bank_dir, n_envs=4, stage_offset=0)
    if start > 393_216:
        bank_dir = resume_run / "banks" / f"seed_{seed}" / "stage_b_to_c"
        return v3_curriculum_vec(
            profile, seed, bank_dir, n_envs=4, stage_offset=5_000_000
        )
    return None

## 11. Deterministic checkpoint evaluation

Full runs evaluate every saved target using the preset DEV seed range and
the same ordinary environment. This writes descriptive simulator metrics
only; it contains no promotion or Final200 decision logic.

In [ ]:
def evaluate_checkpoint(model: Any, target: Any, profile: Path) -> dict[str, Any]:
    evaluation = target.raw["evaluation"]
    start = int(evaluation["dev_seed_start"])
    count = int(evaluation["dev_episodes"])
    physical_seconds = float(evaluation["physical_seconds"])
    floors = []
    env = ordinary_factory(target.id, profile, start)()
    try:
        for episode_seed in range(start, start + count):
            observation, _ = env.reset(seed=episode_seed)
            terminated = truncated = False
            steps = 0
            max_steps = max(1, int(physical_seconds * env.config.fps))
            while not (terminated or truncated) and steps < max_steps:
                action, _ = model.predict(observation, deterministic=True)
                observation, _, terminated, truncated, _ = env.step(
                    int(np.asarray(action).item())
                )
                steps += 1
            floors.append(int(env.simulator.deepest_floor))
    finally:
        env.close()
    return {
        "deterministic": True,
        "floors": floors,
        "metrics": floor_metrics(floors),
    }

## 12. Checkpoints, manifests, Git state, and write protection

Every output is isolated under a unique run directory. Saves are atomic,
output SHA values are metadata, and any destination inside `models/cache`
is rejected before training begins.

In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def git_state() -> tuple[str, bool]:
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
    ).strip()
    status = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=PROJECT_ROOT, text=True
    ).strip()
    return commit, bool(status)


def write_json_atomic(path: Path, payload: Mapping[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + ".tmp")
    partial.write_text(
        json.dumps(dict(payload), ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    partial.replace(path)


def canonical_write_guard(path: Path) -> Path:
    destination = path.resolve()
    protected = {spec.asset_path.resolve() for spec in MODEL_REGISTRY.values()}
    cache_root = (PROJECT_ROOT / "models/cache").resolve()
    if destination in protected or destination.is_relative_to(cache_root):
        raise RuntimeError(f"CANONICAL_MODEL_OVERWRITE_FORBIDDEN:{destination}")
    return destination


def new_run_dir(target: Any, seed: int) -> Path:
    root = canonical_write_guard(OUTPUT_ROOT)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    return root / target.id / f"{target.id}-{TRAINING_MODE}-seed{seed}-{stamp}"


def initialize_run_dir(run_dir: Path, target: Any) -> None:
    run_dir.mkdir(parents=True, exist_ok=False)
    for name in ("checkpoints", "evaluation", "logs"):
        (run_dir / name).mkdir()
    (run_dir / "config_resolved.yaml").write_text(
        yaml.safe_dump(dict(target.raw), sort_keys=False), encoding="utf-8"
    )


def save_checkpoint(model: Any, path: Path) -> tuple[Path, str]:
    canonical_write_guard(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_name(path.stem + ".partial.zip")
    partial.unlink(missing_ok=True)
    model.save(partial)
    if not partial.is_file():
        raise RuntimeError("TRAINING_CHECKPOINT_SAVE_FAILED")
    partial.replace(path)
    return path, sha256_file(path)


def make_training_manifest(
    *, run_id: str, target: Any, commit: str, dirty: bool, device: str,
    seed: int, start: int, target_timesteps: int,
    source_model: str | None, source_sha: str | None,
    assets: list[Mapping[str, Any]], training_performed: str,
) -> dict[str, Any]:
    return {
        "schema_version": "stairkid-training-run-v1",
        "run_id": run_id,
        "training_target": target.id,
        "started_at": utc_now(),
        "completed_at": None,
        "git_commit": commit,
        "git_dirty": dirty,
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "stable_baselines3_version": stable_baselines3.__version__,
        "device": device,
        "seed": int(seed),
        "start_timesteps": int(start),
        "target_timesteps": int(target_timesteps),
        "config_sha256": target.config_sha256,
        "source_model": source_model,
        "source_model_sha256": source_sha,
        "training_assets": [dict(item) for item in assets],
        "output_checkpoint": None,
        "output_sha256": None,
        "observation_space": [268],
        "action_space": "Discrete(3)",
        "training_performed": training_performed,
    }

## 13. Fail-closed precheck

Precheck validates Git state, target/profile, environment spaces, output
protection, and required external assets. It constructs no Real capture or
controller and performs no learning.

In [ ]:
def run_precheck() -> dict[str, Any]:
    commit, dirty = git_state()
    if dirty and not ALLOW_DIRTY:
        raise RuntimeError("TRAINING_REQUIRES_CLEAN_GIT_WORKTREE")
    if not PROFILE_PATH.is_file():
        raise RuntimeError(f"TRAINING_PROFILE_REQUIRED:{PROFILE_PATH}")
    canonical_write_guard(OUTPUT_ROOT)

    env = ordinary_vec(TARGET, PROFILE_PATH, RUN_SEED, smoke=True)
    try:
        if tuple(env.observation_space.shape) != EXPECTED_OBSERVATION_SHAPE:
            raise RuntimeError("TRAINING_ENV_OBSERVATION_MISMATCH")
        if int(env.action_space.n) != EXPECTED_ACTION_COUNT:
            raise RuntimeError("TRAINING_ENV_ACTION_MISMATCH")
    finally:
        env.close()

    assets = []
    if TARGET.id == "r4":
        bundle_path = (PROJECT_ROOT / str(R4_BUNDLE_SPEC["cache_path"])).resolve()
        if not bundle_path.is_file() and TRAINING_MODE != "smoke":
            bundle_path = ensure_r4_bundle()
        available = (
            bundle_path.is_file()
            and sha256_file(bundle_path) == R4_BUNDLE_SHA
        )
        if bundle_path.is_file() and not available:
            raise RuntimeError("R4_TRAINING_BUNDLE_CACHE_SHA_MISMATCH")
        if not available and TRAINING_MODE != "smoke":
            raise RuntimeError("REQUIRED_TRAINING_ASSETS_MISSING:r4_frozen_r1_bundle")
        assets.append({
            "asset_id": "r4_frozen_r1_bundle",
            "sha256": R4_BUNDLE_SHA,
            "available": available,
        })
    return {
        "status": "PASS",
        "git_commit": commit,
        "git_dirty": dirty,
        "target": TARGET.id,
        "config_sha256": TARGET.config_sha256,
        "observation_space": [268],
        "action_space": "Discrete(3)",
        "required_assets": assets,
        "output_root": str(OUTPUT_ROOT),
    }

## 14. Eight-step smoke training

Smoke proves construction, learning, checkpoint save, and manifest output.
V3 starts randomly; R4 copies the canonical R4 policy into a smoke-sized
PPO instance exactly as the reference trainer does. Both canonical model
archives are hashed before and after and must remain unchanged.

In [ ]:
def run_smoke(precheck: Mapping[str, Any]) -> dict[str, Any]:
    canonical_paths = {model_id: ensure_canonical_model(model_id) for model_id in ("v3", "r4")}
    before = {model_id: sha256_file(path) for model_id, path in canonical_paths.items()}
    for model_id, digest in before.items():
        if digest != EXPECTED_MODEL_SHA[model_id]:
            raise RuntimeError(f"CANONICAL_MODEL_SHA_MISMATCH:{model_id}")

    run_dir = new_run_dir(TARGET, RUN_SEED)
    initialize_run_dir(run_dir, TARGET)
    env = ordinary_vec(TARGET, PROFILE_PATH, RUN_SEED, smoke=True)
    source_model = None
    source_sha = None
    try:
        if TARGET.id == "v3":
            model = create_ppo_model(TARGET, env, RUN_SEED, RUN_DEVICE, smoke=True)
            start = 0
        else:
            source_path = canonical_paths["r4"]
            validated = validate_resume_checkpoint(
                source_path, TARGET, env=env, metadata_path=None,
                device=RUN_DEVICE, allow_pinned_external=True,
            )
            model = create_ppo_model(TARGET, env, RUN_SEED, RUN_DEVICE, smoke=True)
            model.policy.load_state_dict(validated.model.policy.state_dict())
            model.num_timesteps = validated.current_timesteps
            start = validated.current_timesteps
            source_model = str(source_path)
            source_sha = validated.sha256

        model.learn(total_timesteps=SMOKE_TIMESTEPS, reset_num_timesteps=False)
        output, output_sha = save_checkpoint(
            model, run_dir / "checkpoints" / f"smoke_{int(model.num_timesteps)}.zip"
        )
    finally:
        env.close()

    after = {model_id: sha256_file(path) for model_id, path in canonical_paths.items()}
    if after != before:
        raise RuntimeError("CANONICAL_MODEL_CHANGED_DURING_SMOKE")
    manifest = make_training_manifest(
        run_id=run_dir.name, target=TARGET, commit=str(precheck["git_commit"]),
        dirty=bool(precheck["git_dirty"]), device=RUN_DEVICE, seed=RUN_SEED,
        start=start, target_timesteps=start + SMOKE_TIMESTEPS,
        source_model=source_model, source_sha=source_sha,
        assets=list(precheck["required_assets"]), training_performed="SMOKE_ONLY",
    )
    manifest.update({
        "completed_at": utc_now(), "output_checkpoint": str(output),
        "output_sha256": output_sha, "actual_timesteps": int(model.num_timesteps),
        "canonical_models_unchanged": True,
    })
    write_json_atomic(run_dir / "training_manifest.json", manifest)
    return {"status": "PASS", "mode": "smoke", "run_dir": str(run_dir), "manifest": manifest}

## 15. Full V3 / R4 training

This is the visible full learning loop. V3 creates its banks at the two
canonical boundaries. R4 stages the frozen R1 bundle and continues in the
ordinary/ordinary/failure/success lanes. `remaining` is the target minus
the checkpoint timestep—not another full target. Every target is saved,
evaluated, and recorded. Nothing runs unless the execution cell is set to
`full` **and** contains the exact authorization phrase.

In [ ]:
def run_full(precheck: Mapping[str, Any]) -> dict[str, Any]:
    if AUTHORIZATION != "AUTHORIZE_STAIRKID_FULL_TRAINING":
        raise RuntimeError("FULL_TRAINING_NOT_AUTHORIZED")
    allowed_seeds = tuple(int(value) for value in TARGET.algorithm["seed_candidates"])
    if RUN_SEED not in allowed_seeds:
        raise RuntimeError("TRAINING_SEED_NOT_IN_PRESET")

    run_dir = new_run_dir(TARGET, RUN_SEED)
    initialize_run_dir(run_dir, TARGET)
    source_model = None
    source_sha = None

    if TARGET.id == "r4":
        stage = stage_and_validate_r4_bundle()
        frozen_source = stage / f"seed_{RUN_SEED}/checkpoints/v3_5_589824.zip"
        frozen_sha = sha256_file(frozen_source)
        bank_dir = stage / f"banks/seed_{RUN_SEED}/r1_targeted"
        env = r4_curriculum_vec(
            PROFILE_PATH, RUN_SEED, bank_dir, frozen_sha,
            n_envs=int(TARGET.algorithm["n_envs"]),
        )
        resume_path = Path(RESUME_CHECKPOINT) if RESUME else frozen_source
        metadata_path = Path(RESUME_METADATA) if RESUME_METADATA else None
        validated = validate_resume_checkpoint(
            resume_path, TARGET, env=env, metadata_path=metadata_path,
            device=RUN_DEVICE, allow_pinned_external=not RESUME,
        )
        model = validated.model
        start = validated.current_timesteps
        source_model = str(validated.path)
        source_sha = validated.sha256
    else:
        env = ordinary_vec(TARGET, PROFILE_PATH, RUN_SEED, smoke=False)
        if RESUME:
            metadata_path = Path(RESUME_METADATA) if RESUME_METADATA else None
            validated = validate_resume_checkpoint(
                Path(RESUME_CHECKPOINT), TARGET, env=env,
                metadata_path=metadata_path, device=RUN_DEVICE,
            )
            model = validated.model
            start = validated.current_timesteps
            source_model = str(validated.path)
            source_sha = validated.sha256
            resumed_env = restore_v3_curriculum_for_resume(
                checkpoint=validated.path, start=start,
                profile=PROFILE_PATH, seed=RUN_SEED,
            )
            if resumed_env is not None:
                env.close()
                env = resumed_env
                model.set_env(env)
        else:
            model = create_ppo_model(TARGET, env, RUN_SEED, RUN_DEVICE, smoke=False)
            start = 0

    manifest = make_training_manifest(
        run_id=run_dir.name, target=TARGET, commit=str(precheck["git_commit"]),
        dirty=bool(precheck["git_dirty"]), device=RUN_DEVICE, seed=RUN_SEED,
        start=start, target_timesteps=TARGET.total_timesteps,
        source_model=source_model, source_sha=source_sha,
        assets=list(precheck["required_assets"]), training_performed="FULL",
    )
    write_json_atomic(run_dir / "training_manifest.json", manifest)

    checkpoint_targets = [
        int(value) for value in TARGET.training["checkpoint_targets"]
        if int(value) > int(model.num_timesteps)
    ]
    latest_output = None
    latest_sha = None
    try:
        for checkpoint_target in checkpoint_targets:
            current = int(model.num_timesteps)
            if TARGET.id == "v3":
                next_env = activate_v3_curriculum_at_boundary(
                    model=model, current=current, run_dir=run_dir,
                    seed=RUN_SEED, profile=PROFILE_PATH, target=TARGET,
                    resume_checkpoint=Path(RESUME_CHECKPOINT) if RESUME else None,
                )
                if next_env is not None:
                    env.close()
                    env = next_env
                    model.set_env(env)

            remaining = checkpoint_target - int(model.num_timesteps)
            quantum = int(TARGET.training["rollout_quantum"])
            if remaining <= 0 or remaining % quantum:
                raise RuntimeError(f"TRAINING_REMAINING_STEPS_INVALID:{remaining}")

            model.learn(total_timesteps=remaining, reset_num_timesteps=False)
            filename = (
                f"fresh_v3_{checkpoint_target}.zip"
                if TARGET.id == "v3"
                else f"v3_5_{checkpoint_target}.zip"
            )
            latest_output, latest_sha = save_checkpoint(
                model, run_dir / "checkpoints" / filename
            )
            write_json_atomic(latest_output.with_suffix(".training.json"), {
                "training_target": TARGET.id,
                "config_sha256": TARGET.config_sha256,
                "num_timesteps": int(model.num_timesteps),
                "output_sha256": latest_sha,
                "policy_seed": RUN_SEED,
            })
            write_json_atomic(
                run_dir / "evaluation" / f"t{checkpoint_target}.json",
                evaluate_checkpoint(model, TARGET, PROFILE_PATH),
            )
    finally:
        env.close()

    manifest.update({
        "completed_at": utc_now(),
        "output_checkpoint": None if latest_output is None else str(latest_output),
        "output_sha256": latest_sha,
        "actual_timesteps": int(model.num_timesteps),
    })
    write_json_atomic(run_dir / "training_manifest.json", manifest)
    return {"status": "PASS", "mode": "full", "run_dir": str(run_dir), "manifest": manifest}

## 16. Execute the selected mode

`Run all` is safe with the default `precheck`. Smoke performs exactly eight
steps. Full training fails before learning unless explicitly authorized.

In [ ]:
if TRAINING_MODE == "full" and AUTHORIZATION != "AUTHORIZE_STAIRKID_FULL_TRAINING":
    raise RuntimeError("FULL_TRAINING_NOT_AUTHORIZED")

PRECHECK_RESULT = run_precheck()
print("REPO_CHECKOUT=PASS")
print(f"GIT_COMMIT={PRECHECK_RESULT['git_commit']}")
print(f"WORKTREE_CLEAN={not PRECHECK_RESULT['git_dirty']}")
print("CONFIG_VALID=PASS")
print(f"TRAIN_TARGET={TRAIN_TARGET}")
print("OBSERVATION_SPACE=(268,)")
print("ACTION_SPACE=Discrete(3)")
print(f"OUTPUT_DIR={OUTPUT_ROOT}")
print("STAIRKID_TRAINING_PRECHECK=PASS")

if TRAINING_MODE == "precheck":
    RESULT = PRECHECK_RESULT
    print("TRAINING_PERFORMED=NO")
elif TRAINING_MODE == "smoke":
    RESULT = run_smoke(PRECHECK_RESULT)
    print("TRAINING_PERFORMED=SMOKE_ONLY")
else:
    RESULT = run_full(PRECHECK_RESULT)
    print("TRAINING_PERFORMED=FULL")

## 17. Result summary

Inspect the resolved commit, target, mode, run directory, checkpoint SHA,
and manifest before copying or sharing an output. Training never updates
the model registry and never changes V3/R4 status.

In [ ]:
print(json.dumps(RESULT, ensure_ascii=False, indent=2))
if TRAINING_MODE != "precheck":
    manifest = RESULT["manifest"]
    print(f"RUN_DIR={RESULT['run_dir']}")
    print(f"OUTPUT_CHECKPOINT={manifest.get('output_checkpoint')}")
    print(f"OUTPUT_SHA256={manifest.get('output_sha256')}")
    print(f"ACTUAL_TIMESTEPS={manifest.get('actual_timesteps')}")
print("REAL_GAME_EXECUTED=NO")
print("CANONICAL_MODEL_PROMOTION=NO")